In [ ]:
import sys
import os
import torch
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx, k_hop_subgraph

# Add project root directory to Python path
sys.path.append(os.path.abspath(".."))
from src.dataset import load_aml_dataset
from src.model import GATv2AMLModel

# ---------------------------------------------------------
# Step 1: Load Data and Model Checkpoint
# ---------------------------------------------------------
data_root = os.path.join("..", "data")
data = load_aml_dataset(root=data_root)

# Initialize GATv2 model architecture
model = GATv2AMLModel(
    in_channels=data.num_node_features,
    hidden_channels=64,
    out_channels=2,
    edge_dim=data.edge_attr.shape[1] if data.edge_attr is not None else 1,
    heads=4
)

# Load saved weights if available
checkpoint_path = os.path.join("..", "checkpoints", "gatv2_best_model.pt")
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
    print("Successfully loaded trained model checkpoint.")
else:
    print("Checkpoint not found; using initialized model weights for demonstration.")

# ---------------------------------------------------------
# Step 2: Extract 2-Hop Subgraph around a Suspicious Account
# ---------------------------------------------------------
# Find node indices flagged as Money Laundering (Class 1)
aml_node_indices = (data.y == 1).nonzero(as_tuple=True)[0]
target_node = aml_node_indices[0].item() if len(aml_node_indices) > 0 else 0

print(f"\nExtracting 2-hop transaction network around Suspicious Account ID: {target_node}")

# Extract subgraph nodes and local edges
subset, sub_edge_index, mapping, edge_mask = k_hop_subgraph(
    node_idx=target_node,
    num_hops=2,
    edge_index=data.edge_index,
    relabel_nodes=True
)

sub_y = data.y[subset].cpu().numpy()

# ---------------------------------------------------------
# Step 3: Plot Suspicious Network Graph using NetworkX
# ---------------------------------------------------------
# Convert PyG subgraph to NetworkX DiGraph
G = nx.DiGraph()
edges = sub_edge_index.t().cpu().numpy()
G.add_edges_from(edges)

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)

# Assign colors: Red for AML Node (1), Skyblue for Normal Node (0)
node_colors = ['#e74c3c' if label == 1 else '#3498db' for label in sub_y]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=600, alpha=0.9)
nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=15, edge_color="gray", width=1.5)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")

plt.title(f"Suspicious Transaction Subgraph (Target Account: {target_node})\nRed = Money Laundering | Blue = Legitimate")
plt.axis("off")
plt.tight_layout()
plt.show()